# Do borrowed ideas stick when grafted locally? — RQ2-D2 host-entry test (demo)

This notebook is a runnable walk-through of the **RQ2-D2 host-entry experiment** (`method.py` + its `src/` stage modules).

**Question.** When a research concept *c* first appears in a subfield *d* that is not its origin subfield (a
*host entry* in year *e*), is the entry followed by uptake from **newcomers** in *d*? And which kind of entry predicts
uptake:

* **Grafting / anchoring (`A_cont`)** — the concept arrives attached to partner concepts that are already *native* to
  the host (partners' pre-entry share of host-subfield papers, from OpenAlex subfield × 5-year-block profiles), or
* **Toolkit / co-transfer (`CT`)** — the concept arrives as a *package* with the companions it had in its origin
  subfield in `[e-5, e-1]`.

**Outcome.** `Y_strict` = number of host papers in the 5 years after entry (`W2 = [e+1, e+5]`) written by
author-disjoint newcomers. **Model.** Poisson pseudo-maximum likelihood (PPML) with high-dimensional fixed effects,
offset `log(n_entry_papers)`, CRV1 standard errors clustered by concept, wild-cluster score bootstrap, Holm correction
over `{b_A, b_CT}`, and a pre-registered decision rule `GRAFTING / TOOLKIT / BOTH / NEITHER`.

**Full-run headline (1,740 screen events, 184 concepts):** primary spec (concept×e + host×e FE) A IRR/SD 1.39
[0.97, 1.99] → NEITHER; it keeps only 26% of events, which triggers the pre-declared fallback that makes the secondary
spec (concept + e + host FE) co-primary: A IRR/SD 1.30 [1.16, 1.45], Holm p ≈ 1e-5 → **GRAFTING**. Co-transfer is null
everywhere.

**What runs here.** Stages 0–6 of `method.py` build the events and W1 features from the hydrated 426-concept
OpenAlex corpus (several GB, not portable to Colab). This demo starts from their output — one row per host-entry event
with W1 features and W2 outcomes, as published in the artifact's `full_method_out.json` — and runs the analysis stages on
a **100-event curated subset** (20 concepts):

| `method.py` stage | what it does | in this notebook |
|---|---|---|
| 0 | freeze the pre-registration (SPEC sha256) | ✅ run, hash compared to the artifact's |
| 1–5 | load corpus, population, events, W1 features | ⏭ replaced by loading `mini_demo_data.json` |
| 6 | outcome sparsity decision | ✅ run (the pre-declared `Y_strict` → `Y_lenient` switch rule) |
| 7 | PPML models + decision | ✅ run (all specs, wild bootstrap, pyfixest cross-check, LPM) |
| 12 | grouped out-of-sample predictive check | ✅ run |
| 8–11, 13–16 | robustness, placebo, labels, power, held-out freeze, figures | ⏭ need the raw corpus / sealed fold |

With 100 events the estimates are noisy — the point is to show the exact pipeline, not to reproduce the full-sample numbers
(those are printed next to the demo numbers at the end).

## Setup
Install dependencies. `pyfixest` and `loguru` are not pre-installed on Colab; the scientific stack is installed at Colab's exact versions only when running outside Colab.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# pyfixest (cross-check + linear probability model), loguru (logging) — NOT on Colab, always install
_pip('pyfixest==0.60.0', 'loguru==0.7.3')

# numpy, pandas, scipy, scikit-learn, matplotlib — pre-installed on Colab, install locally only
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scipy==1.16.3', 'scikit-learn==1.6.1', 'matplotlib==3.10.0')

Imports: the union of the import blocks of `method.py`, `src/config.py`, `src/prereg.py`, `src/ppml.py`, `src/models.py` and `src/assemble_out.py` (minus `orjson`/`pyarrow`, which only served file I/O), plus `matplotlib` for the final plots and `SimpleNamespace` to keep the original `ppml.fit(...)` / `models.X` call sites unchanged inside one notebook.

In [ ]:
from __future__ import annotations

import argparse
import datetime as dt
import hashlib
import json
import math
import os
import sys
import time
import warnings
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
from loguru import logger
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import PoissonRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import matplotlib.pyplot as plt

### Data loading
`mini_demo_data.json` holds 100 screen-fold host-entry events in the artifact's output schema (`input` = JSON of W1 features, `output` = `Y_strict`, `metadata_*` = other outcomes, population, route, field). It is fetched from GitHub, with a local fallback.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-8892b7-occupancy-is-not-integration-for-new/fork/run_WV-8ZZxjzY5t/round-3/experiment-7/demo/mini_demo_data.json"
import json
from pathlib import Path

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    local = Path("mini_demo_data.json")
    if local.exists(): return json.loads(local.read_text())
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(data["metadata"]["method_name"])
print(data["metadata"].get("demo_subset", ""))
print("examples:", len(data["datasets"][0]["examples"]))

## Config
All tunable parameters. The original hard-codes 999 wild-bootstrap draws (`models.fit_one`) and 1,000 concept-bootstrap
draws (`assemble_out.oos`); both are cheap on 100 events, so the demo uses the original values. `N_EVENTS` caps the
number of events taken from the demo file (all 100; the full run used 1,740 events from 184 concepts).

In [ ]:
N_EVENTS = 100            # events used from mini_demo_data.json (max 100; full run: 1740)
WILD_REPS = 999           # wild cluster score bootstrap draws per coefficient (original: 999)
OOS_BOOT_REPS = 1000       # concept bootstrap draws for the out-of-sample deviance CI (original: 1000)
OOS_FOLDS = 5             # grouped-by-concept folds for the out-of-sample check (original: 5)
RUN_PYFIXEST_CROSSCHECK = True  # stage 7 `crosscheck=True` in method.py: refit M1 with pyfixest.fepois
PPML_MAXIT, PPML_TOL = 500, 1e-12  # IRLS settings used by models.fit_one (original values)

## `src/config.py` — the frozen pre-registration SPEC
Every decision of the analysis (event definition, nativeness blocks, outcomes, FE specs, Holm family, decision rule,
fallbacks) is written into `SPEC` *before* any outcome is computed; its sha256 is the pre-registration hash.
The corpus/dependency paths, RAM limit and held-out sealing guard of the original `config.py` are omitted (they only
guard raw-corpus stages); `RESULTS`/`LOGS` point at local folders.

In [ ]:
RESULTS = Path("results")
LOGS = Path("logs")
for _d in (RESULTS, LOGS):
    _d.mkdir(parents=True, exist_ok=True)

SEED = 20260929

SPEC: dict = dict(
    name="RQ2-D2 host-entry anchoring (grafting) vs co-transfer (toolkit)",
    event=dict(
        known_sub_min_topic_score=0.05, e_max=2019, main_rule="F <= e <= 2019",
        reference_rule="2005 <= e <= 2019 and no d-paper in 2000-2004", kw5_min_distinct_partners=5,
        dedup="within (concept, dup_group): earliest year, then max n_refs, then min work_id (iteration-1 rule)",
        year_field="concept_work link year"),
    partners=dict(source="legacy OpenAlex concept tags of the entry-year d-papers", min_score=0.2, min_level=1,
                  drop_own_linked_concept=True, weighting="tag-weighted multiset over (paper, partner) pairs"),
    nativeness=dict(blocks={"2005-2009": "2000-2004", "2010-2014": "2005-2009", "2015-2019": "2010-2014"},
                    share="counts[d] / (total - unknown); truncated_top200 and d absent -> 0",
                    native_threshold=0.5, threshold_curve=[0.3, 0.7]),
    co_transfer=dict(companions="partners on c-papers with sub == o(c) in [e-5, e-1]",
                     variant_any="partners on any earlier c-paper (years < e)"),
    outcomes=dict(W2="[e+1, e+5]", primary="Y_strict", prior_set="authors of c-papers in [e-5, e] plus their "
                  "co-authors on any corpus work in [e-5, e]", Y_strict="W2 d-papers of c with no author in Pset "
                  "(papers without author ids excluded)", Y_lenient="W2 d-papers with >= 1 author not in Pset",
                  Y_all="all W2 d-papers of c", EST_bin="Y_strict >= 3 and d-papers in >= 3 of the 5 W2 years",
                  zero_share_switch=0.85),
    models=dict(estimator="PPML (IRLS, exact sparse FE projection, singleton/separation pruning)",
                primary_fe=["concept x e", "d x e"], offset="log(n_entry_papers)", cluster="concept",
                primary_controls=["prox_od", "RD", "log_n_partner_tags", "cov", "demic", "mean_topic_score",
                                  "boundary_share", "abstract_share"],
                secondary_fe=["concept", "e", "d"],
                secondary_controls=["prox_od", "RD", "log_n_partner_tags", "cov", "demic", "mean_topic_score",
                                    "boundary_share", "abstract_share", "mom_d", "log_centrality", "log_W1",
                                    "route_A"],
                sample="screen fold, MAIN population, kw5, main arm (F <= e <= 2019)",
                coefficient_scale="IRR per 0.1 of A / CT and per SD",
                inference="CRV1 by concept (pyfixest small-sample factor); wild cluster score bootstrap p (999)",
                holm_family=["b_A", "b_CT"],
                decision={"GRAFTING": "b_A > 0, Holm p < .05, b_CT not significantly > 0",
                          "TOOLKIT": "b_CT > 0, Holm p < .05, b_A n.s.",
                          "BOTH": "both significantly > 0", "NEITHER": "otherwise (report MDE)",
                          "negative": "a significant negative sign is reported as such"},
                fallback_thin_cells="retained < 30% of events or G < 50 -> secondary spec co-primary"),
    placebo=dict(draws=500, label_shuffle_draws=50),
    bootstrap_reps=1000, wild_reps=999, power_reps=300, power_grid=[1.00, 1.05, 1.10, 1.15, 1.20, 1.30, 1.40],
    label=dict(benchmark="median entry A over REFERENCE_ACCEPTED reference-arm events in d (>= 5), else field, "
               "else global", lower_bound="bootstrap 90% lower (n_entry_papers >= 3) else Clopper-Pearson 90% "
               "lower"),
    fold_rule="screen iff int(sha1(concept_id), 16) % 10 < 7",
    seed=SEED,
)


def spec_hash() -> str:
    return hashlib.sha256(json.dumps(SPEC, sort_keys=True).encode()).hexdigest()


def setup_logging(name: str) -> None:
    logger.remove()
    logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{name}:{line}|{message}")
    logger.add(LOGS / f"{name}.log", rotation="30 MB", level="DEBUG")


def fold_of(concept_id: str) -> str:
    return "screen" if int(hashlib.sha1(concept_id.encode()).hexdigest(), 16) % 10 < 7 else "heldout"


setup_logging("method")

## Stage 0 — `src/prereg.py`: freeze the pre-registration
`freeze()` writes `results/d2_prereg.json` with the SPEC hash (or refuses if the SPEC changed after an earlier freeze).
Because the SPEC above is copied verbatim, its hash must equal the `prereg_sha256` recorded in the artifact output
(`f800a0a9…`) — a check that the notebook runs the pre-registered analysis.

In [ ]:
def freeze() -> str:
    p = RESULTS / "d2_prereg.json"
    h = spec_hash()
    if p.exists():
        old = json.loads(p.read_text())
        if old["sha256"] != h:
            raise RuntimeError("SPEC changed after the freeze: log the change in results/deviations.md and re-freeze "
                               "explicitly (delete results/d2_prereg.json) if it is intended")
        logger.info(f"prereg already frozen: {h}")
        return h
    ts = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    p.write_text(json.dumps({"spec": SPEC, "sha256": h, "frozen_utc": ts,
                             "note": "sha256 = sha256(json.dumps(spec, sort_keys=True)); frozen before stage 6 "
                                     "(outcomes) ran for the first time"}, indent=1))
    with open(LOGS / "prereg_freeze.log", "a") as f:
        f.write(f"{h} {ts}\n")
    logger.info(f"prereg frozen {h} at {ts}")
    return h


t0 = time.time()
timings = {}
h = freeze()
print("SPEC sha256 here     :", h)
print("prereg_sha256 of run :", data["metadata"]["prereg_sha256"])
print("identical:", h == data["metadata"]["prereg_sha256"])

## Stages 1–5 (replaced) — rebuild the event table from the demo data
In `method.py`, stages 1–6 load the corpus, build the MAIN population, detect host-entry events, compute W1 features
(`A_cont`, `CT`, controls) and W2 outcomes, and save `results/screen_events_with_outcomes.parquet`. Here we rebuild the
columns stage 7 reads from that parquet out of the demo JSON. All demo events are, by construction, **main arm, screen
fold, MAIN population, ≥5 distinct partners (`kw5`)** — the primary sample. `route_A` (retrieval via OpenAlex-native
concept, a secondary control) is recovered from `metadata_route`.

In [ ]:
rows = []
for ex in data["datasets"][0]["examples"][:N_EVENTS]:
    r = json.loads(ex["input"])                      # W1 features of the event (concept c, origin o, host d, year e)
    r.update(arm="main", fold=ex["metadata_fold"], kw5=True, MAIN=ex["metadata_population"].startswith("MAIN"),
             STRICT="STRICT" in ex["metadata_population"], route=ex["metadata_route"],
             route_A=ex["metadata_route"] == "A_openalex_native", field_group=ex["metadata_field_group"],
             Y_strict=int(ex["output"]), Y_lenient=ex["metadata_Y_lenient"], Y_all=ex["metadata_Y_all"],
             EST_bin=ex["metadata_EST_bin"])
    rows.append(r)
df_events = pd.DataFrame(rows)
assert (df_events.concept_id.map(fold_of) == df_events.fold).all()  # screen-fold rule from SPEC
print(df_events.shape, "| concepts:", df_events.concept_id.nunique(), "| hosts d:", df_events.d.nunique(),
      "| entry years:", sorted(df_events.e.unique()))
df_events[["concept_id", "o", "d", "e", "A_cont", "CT", "n_entry_papers", "Y_strict", "Y_lenient", "Y_all",
           "EST_bin", "field_group"]].head()

## Stage 6 — outcome sparsity decision (from `method.py`)
Before any coefficient is estimated, the pre-registered rule checks whether `Y_strict` is too sparse: if more than 85%
of primary-sample events have `Y_strict = 0`, the primary outcome switches to `Y_lenient`. (The outcome computation
itself, `outcomes.compute_Y`, needs the raw corpus; its results are already in the event table.)

In [ ]:
t = time.time()
out = df_events
p = out[out.MAIN & out.kw5]
dec = {"n_primary_sample": int(len(p)), "Y_strict_zero_share": float((p.Y_strict == 0).mean()),
       "Y_lenient_zero_share": float((p.Y_lenient == 0).mean()),
       "Y_all_zero_share": float((p.Y_all == 0).mean()),
       "rule": "switch primary to Y_lenient iff Y_strict zero share > 0.85 (decided before any coefficient)"}
dec["switch_to_lenient"] = dec["Y_strict_zero_share"] > 0.85
(RESULTS / "outcome_sparsity_decision.json").write_text(json.dumps(dec, indent=1))
timings[6] = round(time.time() - t, 1)
print(json.dumps(dec, indent=1))

## `src/ppml.py` — fast PPML with high-dimensional fixed effects
* `_prune` iteratively drops singleton FE levels and levels whose outcome is all zero (separation) — this is why the
  concept×e primary spec keeps only ~26% of events in the full run.
* `_demean` projects variables off the FE space exactly (sparse LU of `D'WD`).
* `fit` runs IRLS on the demeaned working variable and returns CRV1 (cluster = concept) standard errors with the
  pyfixest small-sample factor.
* `wild_score_test` is the Kline–Santos wild cluster (Rademacher) score bootstrap for `H0: b = 0`.

Code copied verbatim.

In [ ]:
def _prune(y: np.ndarray, fes: list[np.ndarray]) -> np.ndarray:
    """Iteratively drop singleton FE levels and FE levels with all-zero outcome (separation)."""
    keep = np.ones(len(y), bool)
    for _ in range(50):
        changed = False
        for f in fes:
            cnt = np.bincount(f[keep], minlength=f.max() + 1)
            sy = np.bincount(f[keep], weights=y[keep], minlength=f.max() + 1)
            bad = (cnt <= 1) | (sy <= 0)
            drop = keep & bad[f]
            if drop.any():
                keep &= ~drop
                changed = True
        if not changed:
            break
    return keep


def _demean(M: np.ndarray, w: np.ndarray, fes: list[np.ndarray]) -> np.ndarray:
    """Exact weighted projection off the FE space: solve (D'WD + eps I) c = D'W M with a sparse LU (D = [D1 D2])."""
    from scipy import sparse
    from scipy.sparse.linalg import splu
    n = len(w)
    rows, cols, off = [], [], 0
    for f in fes:
        rows.append(np.arange(n))
        cols.append(f + off)
        off += int(f.max()) + 1
    D = sparse.csc_matrix((np.ones(n * len(fes)), (np.concatenate(rows), np.concatenate(cols))), shape=(n, off))
    DW = sparse.csc_matrix(D.multiply(w[:, None]))
    A = sparse.csc_matrix(D.T @ DW) + sparse.identity(off, format="csc") * 1e-9
    c = splu(A).solve(np.asarray(DW.T @ M))
    return M - D @ c


def pd_nunique_per_level(f: np.ndarray, g: np.ndarray) -> int:
    """Max number of distinct clusters within any FE level."""
    pairs = np.unique(np.stack([f, g], 1), axis=0)
    return int(np.bincount(pairs[:, 0]).max())


def fit(y: np.ndarray, X: np.ndarray, fes: list[np.ndarray], offset: np.ndarray, cluster: np.ndarray,
        maxit: int = 100, tol: float = 1e-9) -> dict | None:
    keep = _prune(y.astype(float), fes)
    if keep.sum() < X.shape[1] + 5:
        return None
    y, X, off, cl = y[keep].astype(float), X[keep], offset[keep], cluster[keep]
    fes = [np.unique(f[keep], return_inverse=True)[1] for f in fes]
    if np.linalg.matrix_rank(_demean(X, np.ones(len(y)), fes)) < X.shape[1]:
        return None
    mu = np.maximum(y, 0.1 * y.mean() + 1e-3)
    eta = np.log(mu)
    beta = np.zeros(X.shape[1])
    dev_old = np.inf
    for _ in range(maxit):
        z = eta - off + (y - mu) / mu
        M = _demean(np.column_stack([z, X]), mu, fes)
        zt, Xt = M[:, 0], M[:, 1:]
        WX = Xt * mu[:, None]
        beta = np.linalg.solve(Xt.T @ WX, WX.T @ zt)
        resid = zt - Xt @ beta
        eta = z - resid + off  # eta = X b + FE + offset
        eta = np.clip(eta, -30, 30)
        mu = np.exp(eta)
        dev = 2 * np.sum(np.where(y > 0, y * np.log(np.where(y > 0, y, 1) / mu), 0) - (y - mu))
        if abs(dev - dev_old) / (abs(dev) + 0.1) < tol:
            break
        dev_old = dev
    Xt = _demean(X, mu, fes)
    H = Xt.T @ (Xt * mu[:, None])
    Hi = np.linalg.inv(H)
    sc = Xt * (y - mu)[:, None]
    _, g = np.unique(cl, return_inverse=True)
    G = g.max() + 1
    S = np.zeros((G, X.shape[1]))
    np.add.at(S, g, sc)
    n = len(y)
    # small-sample factor as pyfixest CRV1 (fixef_k='nested'): FE levels nested in clusters are not counted
    k_fe = 0
    for f in fes:
        nested = np.all(np.bincount(f, minlength=f.max() + 1) == 0) or (
            pd_nunique_per_level(f, g) <= 1)
        if not nested:
            k_fe += int(f.max()) + 1 - 1
    k = X.shape[1] + k_fe + (1 if k_fe else 0)
    adj = G / (G - 1) * (n - 1) / (n - k) if G > 1 and n > k else 1.0
    V = adj * Hi @ (S.T @ S) @ Hi
    return {"coef": beta, "se": np.sqrt(np.diag(V)), "n": int(n), "G": int(G), "mu": mu, "keep": keep,
            "Xt": Xt, "y": y, "cl": g}


def wild_score_test(y, X_restricted, x2, fes, offset, cluster, rng, reps: int = 199) -> float | None:
    """Kline-Santos wild cluster (Rademacher) score bootstrap for H0: coefficient on x2 = 0."""
    r = fit(y, X_restricted, fes, offset, cluster)
    if r is None:
        return None
    keep, mu = r["keep"], r["mu"]
    fes_k = [np.unique(f[keep], return_inverse=True)[1] for f in fes]
    Xall = np.column_stack([x2[keep], X_restricted[keep]])
    D = _demean(Xall, mu, fes_k)
    x2t, Xrt = D[:, 0], D[:, 1:]
    b = np.linalg.solve(Xrt.T @ (Xrt * mu[:, None]), (Xrt * mu[:, None]).T @ x2t)
    x2r = x2t - Xrt @ b
    s = np.bincount(r["cl"], weights=x2r * (r["y"] - mu))
    if len(s) < 2 or (s ** 2).sum() == 0:
        return None
    T = s.sum() ** 2 / (s ** 2).sum()
    eps = rng.choice([-1.0, 1.0], size=(reps, len(s)))
    Ts = (eps @ s) ** 2 / (s ** 2).sum()
    return float((1 + (Ts >= T).sum()) / (reps + 1))


# notebook shim: the original modules call these as ppml.fit(...) / ppml.wild_score_test(...)
ppml = SimpleNamespace(fit=fit, wild_score_test=wild_score_test, _prune=_prune, _demean=_demean)

## `src/models.py` — sample, FE specs, fits and the pre-registered decision
* `primary_sample`: screen fold, MAIN, kw5, main arm, complete controls; builds the FE codes
  (`cxe` = concept×e, `dxe` = host×e, `cfe`/`efe`/`dfe` = concept / e / host, `cx2` = concept×2-year bin) and the
  offset `log(n_entry_papers)`.
* `FE_SPECS`: **primary** (concept×e + host×e), **secondary** (concept + e + host; co-primary under fallback 3),
  and two pre-declared alternatives.
* `fit_one`: one PPML fit → coefficients, CRV1 p (t with G−1 df), IRR per SD and per 0.1, 95% CI of IRR/SD,
  optional wild-bootstrap p. **Only change:** the hard-coded `reps=999` becomes the config variable `WILD_REPS`
  (and `maxit/tol` read `PPML_MAXIT/PPML_TOL`, same values).
* `holm` / `decide`: Holm over `{b_A, b_CT}` and the GRAFTING / TOOLKIT / BOTH / NEITHER rule.
* `pyfixest_crosscheck` refits M1 with `pyfixest.fepois`; `lpm_fe` is the linear probability model for `EST_bin`
  (establishment: `Y_strict ≥ 3` spread over ≥ 3 of the 5 W2 years).
* `run_models`: all models of stage 7 plus the thin-cells fallback check.

In [ ]:
A_VAR = "A_cont"  # primary anchoring measure after the pre-declared nativeness fallback (deviations.md #1)
CT_VAR = "CT"
EVENT_CONTROLS = ["prox_od", "RD", "log_n_partner_tags", "cov", "demic", "mean_topic_score", "boundary_share",
                  "abstract_share"]
SECONDARY_EXTRA = ["mom_d", "log_centrality", "log_W1"]


def primary_sample(df: pd.DataFrame, pop: str = "MAIN", fold: str = "screen") -> pd.DataFrame:
    s = df[(df.arm == "main") & (df.fold == fold) & df.kw5 & df[pop]].copy()
    need = [A_VAR, CT_VAR] + EVENT_CONTROLS + SECONDARY_EXTRA
    s = s.dropna(subset=[c for c in need if c in s.columns])
    s["cxe"] = pd.factorize(s.concept_id + "_" + s.e.astype(str))[0]
    s["dxe"] = pd.factorize(s.d.astype(str) + "_" + s.e.astype(str))[0]
    s["cfe"] = pd.factorize(s.concept_id)[0]
    s["efe"] = pd.factorize(s.e.astype(str))[0]
    s["dfe"] = pd.factorize(s.d.astype(str))[0]
    s["cx2"] = pd.factorize(s.concept_id + "_" + ((s.e - 2005) // 2).astype(str))[0]
    s["offset"] = np.log(s.n_entry_papers.astype(float))
    return s.reset_index(drop=True)


FE_SPECS = {"primary": ["cxe", "dxe"], "secondary": ["cfe", "efe", "dfe"],
            "coarse_cx2y": ["cx2", "dxe"],  # pre-declared coarsening: concept x 2-year entry bin + d x e
            "c_plus_dxe": ["cfe", "dxe"]}   # pre-declared alternative: concept + d x e


def fe_arrays(s: pd.DataFrame, spec: str) -> list[np.ndarray]:
    return [s[c].to_numpy() for c in FE_SPECS[spec]]


def fit_one(s: pd.DataFrame, y: str, xvars: list[str], spec: str = "primary", wild: bool = False,
            wild_vars: tuple = (), rng=None) -> dict | None:
    X = s[xvars].to_numpy(float)
    fes = fe_arrays(s, spec)
    yv = s[y].to_numpy(float)
    r = ppml.fit(yv, X, fes, s.offset.to_numpy(), s.concept_id.to_numpy(), maxit=PPML_MAXIT, tol=PPML_TOL)
    if r is None:
        return None
    keep = r["keep"]
    G = r["G"]
    out = {"spec": spec, "y": y, "x": xvars, "n_input": int(len(s)), "n_retained": int(r["n"]), "G": int(G),
           "retained_share": float(r["n"] / len(s)), "coef": {}, "se": {}, "p": {}, "irr_sd": {}, "irr_01": {},
           "ci_irr_sd": {}, "sd_retained": {}}
    # singleton / separation accounting for the first FE dimension
    cells = pd.Series(fes[0]).value_counts()
    out["n_nonsingleton_fe0_cells"] = int((cells >= 2).sum())
    out["n_fe0_cells"] = int(len(cells))
    out["events_in_nonsingleton_fe0_cells"] = int(cells[cells >= 2].sum())
    tq = stats.t.ppf(0.975, G - 1)
    for i, v in enumerate(xvars):
        b, se = float(r["coef"][i]), float(r["se"][i])
        sd = float(s.loc[keep, v].std())
        tstat = b / se if se > 0 else np.nan
        out["coef"][v], out["se"][v] = b, se
        out["p"][v] = float(2 * stats.t.sf(abs(tstat), G - 1)) if se > 0 else np.nan
        out["sd_retained"][v] = sd
        out["irr_sd"][v] = float(np.exp(b * sd))
        out["irr_01"][v] = float(np.exp(b * 0.1))
        out["ci_irr_sd"][v] = [float(np.exp((b - tq * se) * sd)), float(np.exp((b + tq * se) * sd))]
    out["deviance"] = float(2 * np.sum(np.where(r["y"] > 0, r["y"] * np.log(np.where(r["y"] > 0, r["y"], 1) / r["mu"]),
                                                 0) - (r["y"] - r["mu"])))
    if wild:
        rng = rng or np.random.default_rng(SEED)
        out["p_wild"] = {}
        for v in wild_vars:
            j = xvars.index(v)
            Xr = np.delete(X, j, axis=1)
            out["p_wild"][v] = ppml.wild_score_test(yv, Xr, X[:, j], fes, s.offset.to_numpy(), s.concept_id.to_numpy(),
                                                    rng, reps=WILD_REPS)
    return out


def holm(pvals: dict) -> dict:
    items = sorted(pvals.items(), key=lambda kv: kv[1])
    m = len(items)
    adj, run = {}, 0.0
    for i, (k, p) in enumerate(items):
        run = max(run, min(1.0, (m - i) * p))
        adj[k] = run
    return adj


def decide(res: dict) -> dict:
    bA, bC = res["coef"][A_VAR], res["coef"][CT_VAR]
    h = holm({A_VAR: res["p"][A_VAR], CT_VAR: res["p"][CT_VAR]})
    sigA, sigC = h[A_VAR] < 0.05, h[CT_VAR] < 0.05
    if sigA and bA > 0 and not (sigC and bC > 0):
        reading = "GRAFTING"
    elif sigC and bC > 0 and not sigA:
        reading = "TOOLKIT"
    elif sigA and bA > 0 and sigC and bC > 0:
        reading = "BOTH"
    else:
        reading = "NEITHER"
    notes = []
    if sigA and bA < 0:
        notes.append("significant NEGATIVE anchoring coefficient (anchoring penalty)")
    if sigC and bC < 0:
        notes.append("significant NEGATIVE co-transfer coefficient (package penalty)")
    if sigC and bC > 0 and sigA and bA < 0:
        notes.append("co-transfer premium with an anchoring penalty")
    return {"reading": reading, "holm_p": h, "notes": notes}


def pyfixest_crosscheck(s: pd.DataFrame, y: str, xvars: list[str]) -> dict:
    """pyfixest.fepois on the sample retained by our iterated singleton/separation pruning, tight tolerances.
    (pyfixest's own pruning is not iterated, so on the unpruned sample it keeps extra singletons: same coefficients,
    different N and small-sample factor.)"""
    import pyfixest as pf
    r_own = fit_one(s, y, xvars, "primary")
    r_raw = ppml.fit(s[y].to_numpy(float), s[xvars].to_numpy(float), fe_arrays(s, "primary"), s.offset.to_numpy(),
                     s.concept_id.to_numpy())
    d = s[r_raw["keep"]].copy()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        m = pf.fepois(f"{y} ~ {' + '.join(xvars)} | cxe + dxe", data=d, offset="offset", vcov={"CRV1": "concept_id"},
                      demeaner_backend="scipy", fixef_tol=1e-12, iwls_tol=1e-12, iwls_maxiter=500)
    cf, se = m.coef(), m.se()
    diffs = {v: {"coef_own": r_own["coef"][v], "coef_pf": float(cf[v]), "abs_diff": abs(r_own["coef"][v] - float(cf[v])),
                 "se_own": r_own["se"][v], "se_pf": float(se[v]),
                 "se_rel_diff": abs(r_own["se"][v] - float(se[v])) / float(se[v])} for v in xvars}
    ok_coef = all(x["abs_diff"] < 1e-4 for x in diffs.values())
    ok_se = all(x["se_rel_diff"] < 1e-3 for x in diffs.values())
    return {"pass_coef_1e-4": ok_coef, "pass_se_rel_1e-3": ok_se, "n_pf": int(m._N), "per_var": diffs}


def lpm_fe(s: pd.DataFrame, y: str, xvars: list[str], spec: str = "primary") -> dict:
    import pyfixest as pf
    fe = "cxe + dxe" if spec == "primary" else "cfe + efe + dfe"
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        m = pf.feols(f"{y} ~ {' + '.join(xvars)} | {fe}", data=s, vcov={"CRV1": "concept_id"})
    return {"spec": spec, "y": y, "model": "linear probability, same FE", "n": int(m._N),
            "coef": {v: float(m.coef()[v]) for v in [A_VAR, CT_VAR] if v in xvars},
            "se": {v: float(m.se()[v]) for v in [A_VAR, CT_VAR] if v in xvars},
            "p": {v: float(m.pvalue()[v]) for v in [A_VAR, CT_VAR] if v in xvars},
            "effect_per_sd": {v: float(m.coef()[v] * s[v].std()) for v in [A_VAR, CT_VAR] if v in xvars}}


def run_models(df: pd.DataFrame, crosscheck: bool = False) -> dict:
    s = primary_sample(df)
    ctrl = EVENT_CONTROLS
    rng = np.random.default_rng(SEED)
    res = {"sample": {"n_events": int(len(s)), "n_concepts": int(s.concept_id.nunique()),
                      "rule": "screen fold, MAIN, kw5, main arm, complete controls",
                      "dropped_missing_controls": int(len(df[(df.arm == "main") & (df.fold == "screen") & df.kw5
                                                                & df.MAIN]) - len(s))}}
    res["M0"] = fit_one(s, "Y_strict", ctrl, "primary")
    res["M1"] = fit_one(s, "Y_strict", [A_VAR, CT_VAR] + ctrl, "primary", wild=True, wild_vars=(A_VAR, CT_VAR), rng=rng)
    res["M1a"] = fit_one(s, "Y_strict", [A_VAR] + ctrl, "primary")
    res["M1b"] = fit_one(s, "Y_strict", [CT_VAR] + ctrl, "primary")
    res["M1_secondary"] = fit_one(s, "Y_strict", [A_VAR, CT_VAR] + ctrl + SECONDARY_EXTRA, "secondary", wild=True,
                                  wild_vars=(A_VAR, CT_VAR), rng=rng)
    res["M0_secondary"] = fit_one(s, "Y_strict", ctrl + SECONDARY_EXTRA, "secondary")
    res["M1a_secondary"] = fit_one(s, "Y_strict", [A_VAR] + ctrl + SECONDARY_EXTRA, "secondary")
    res["M1b_secondary"] = fit_one(s, "Y_strict", [CT_VAR] + ctrl + SECONDARY_EXTRA, "secondary")
    res["M1_coarse_cx2y"] = fit_one(s, "Y_strict", [A_VAR, CT_VAR] + ctrl, "coarse_cx2y", wild=True,
                                    wild_vars=(A_VAR, CT_VAR), rng=rng)
    res["M1_c_plus_dxe"] = fit_one(s, "Y_strict", [A_VAR, CT_VAR] + ctrl, "c_plus_dxe", wild=True,
                                   wild_vars=(A_VAR, CT_VAR), rng=rng)
    for y in ["Y_all", "Y_lenient"]:
        res[f"M1_{y}"] = fit_one(s, y, [A_VAR, CT_VAR] + ctrl, "primary")
        res[f"M1_{y}_secondary"] = fit_one(s, y, [A_VAR, CT_VAR] + ctrl + SECONDARY_EXTRA, "secondary")
    res["EST_bin_LPM"] = lpm_fe(s, "EST_bin", [A_VAR, CT_VAR] + ctrl, "primary")
    res["EST_bin_LPM_secondary"] = lpm_fe(s, "EST_bin", [A_VAR, CT_VAR] + ctrl + SECONDARY_EXTRA, "secondary")
    m1 = res["M1"]
    thin = m1 is None or m1["retained_share"] < 0.30 or m1["G"] < 50
    res["fallback3_thin_cells"] = {"triggered": bool(thin),
                                   "retained_share": None if m1 is None else m1["retained_share"],
                                   "G": None if m1 is None else m1["G"],
                                   "rule": "retained < 30% of events or G < 50 -> secondary spec is co-primary"}
    res["decision_primary"] = decide(m1) if m1 else None
    res["decision_secondary"] = decide(res["M1_secondary"]) if res["M1_secondary"] else None
    if thin:
        res["decision_note"] = ("within concept x entry-year identification is thin: the secondary spec is "
                                "co-primary (pre-declared fallback 3)")
    for k in ["M1_coarse_cx2y", "M1_c_plus_dxe"]:
        res[f"decision_{k}"] = decide(res[k]) if res[k] else None
    cc = RESULTS / "pyfixest_crosscheck.json"
    if crosscheck or not cc.exists():
        try:
            cc.write_text(json.dumps(pyfixest_crosscheck(s, "Y_strict", [A_VAR, CT_VAR] + ctrl), indent=1, default=float))
        except Exception as ex:  # noqa: BLE001 - record and continue; the own estimator is validated in tests
            cc.write_text(json.dumps({"error": repr(ex)}))
    res["pyfixest_crosscheck"] = json.loads(cc.read_text())
    (RESULTS / "d2_models.json").write_text(json.dumps(res, indent=1, default=float))
    rows = []
    for k, v in res.items():
        if isinstance(v, dict) and "coef" in v and "x" in v:
            for var in [A_VAR, CT_VAR]:
                if var in v["coef"]:
                    rows.append({"model": k, "spec": v["spec"], "y": v["y"], "var": var, "coef": v["coef"][var],
                                 "se": v["se"][var], "p": v["p"][var], "irr_sd": v["irr_sd"][var],
                                 "irr_sd_lo": v["ci_irr_sd"][var][0], "irr_sd_hi": v["ci_irr_sd"][var][1],
                                 "irr_01": v["irr_01"][var], "n": v["n_retained"], "G": v["G"],
                                 "p_wild": (v.get("p_wild") or {}).get(var)})
    pd.DataFrame(rows).to_csv(RESULTS / "d2_models_table.csv", index=False)
    logger.info(f"M1: n={m1['n_retained']} G={m1['G']} bA={m1['coef'][A_VAR]:.3f} (p {m1['p'][A_VAR]:.4f}) "
                f"bCT={m1['coef'][CT_VAR]:.3f} (p {m1['p'][CT_VAR]:.4f}); decision {res['decision_primary']}")
    return res


# notebook shim: assemble_out.py refers to these as models.X
models = SimpleNamespace(primary_sample=primary_sample, fit_one=fit_one, fe_arrays=fe_arrays,
                         EVENT_CONTROLS=EVENT_CONTROLS, SECONDARY_EXTRA=SECONDARY_EXTRA)

## Stage 7 — run the PPML models
Exactly the stage-7 branch of `method.py`: `models.run_models(<screen events with outcomes>, crosscheck=True)`.
It writes `results/d2_models.json`, `results/d2_models_table.csv` and `results/pyfixest_crosscheck.json`.

In [ ]:
t = time.time()
res = run_models(df_events, crosscheck=RUN_PYFIXEST_CROSSCHECK)
timings[7] = round(time.time() - t, 1)
logger.info(f"stage 7 done in {timings[7]} s")
print("sample:", res["sample"])
print("fallback 3 (thin cells):", res["fallback3_thin_cells"])
for k in ["decision_primary", "decision_secondary", "decision_M1_coarse_cx2y", "decision_M1_c_plus_dxe"]:
    print(f"{k:26s}", res[k])
cc = res["pyfixest_crosscheck"]
print("pyfixest cross-check:", {k: v for k, v in cc.items() if k != "per_var"})

## Stage 12 — `src/assemble_out.py`: grouped out-of-sample predictive check
A *descriptive* check (not the inferential test, since a concept FE cannot predict unseen concepts): events are split into
5 folds by `sha1(concept_id) % 5`; on 4 folds a Poisson GLM with year and host one-hot FE, concept-level covariates and
exposure `n_entry_papers` is fitted as **M0** (controls only) and **M1** (M0 + `A_cont` + `CT`), and `Y_strict` is
predicted on the held-back fold. The deviance difference M1 − M0 gets a concept-bootstrap CI.
Code copied verbatim; the only changes are `range(1000)` → `range(OOS_BOOT_REPS)` and `% 5` / `range(5)` →
`OOS_FOLDS` (same value). Writing `method_out.json` (needs stage-10 graft labels) is skipped.

In [ ]:
NUM0 = models.EVENT_CONTROLS + models.SECONDARY_EXTRA + ["route_A_f"]
NUM1 = NUM0 + ["A_cont", "CT"]


def _model(num: list[str]):
    ct = ColumnTransformer([("num", StandardScaler(), num),
                            ("fe", OneHotEncoder(handle_unknown="ignore"), ["e_s", "d_s"])])
    return make_pipeline(ct, PoissonRegressor(alpha=1e-4, max_iter=3000))


def poisson_dev(y: np.ndarray, mu: np.ndarray) -> np.ndarray:
    return 2 * (np.where(y > 0, y * np.log(np.where(y > 0, y, 1) / mu), 0) - (y - mu))


def oos(s: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    s = s.copy()
    s["route_A_f"] = s.route_A.astype(float)
    s["e_s"], s["d_s"] = s.e.astype(str), s.d.astype(str)
    s["k"] = [int(hashlib.sha1(c.encode()).hexdigest(), 16) % OOS_FOLDS for c in s.concept_id]
    s["rate"] = s.Y_strict / s.n_entry_papers
    s["mu0"], s["mu1"] = np.nan, np.nan
    for k in range(OOS_FOLDS):
        tr, te = s.k != k, s.k == k
        for name, num in (("mu0", NUM0), ("mu1", NUM1)):
            m = _model(num)
            m.fit(s.loc[tr], s.loc[tr, "rate"], poissonregressor__sample_weight=s.loc[tr, "n_entry_papers"])
            s.loc[te, name] = m.predict(s.loc[te]) * s.loc[te, "n_entry_papers"]
    y = s.Y_strict.to_numpy(float)
    d0, d1 = poisson_dev(y, s.mu0.to_numpy()), poisson_dev(y, s.mu1.to_numpy())
    rng = np.random.default_rng(SEED)
    cids = s.concept_id.to_numpy()
    uc = np.unique(cids)
    grp = {c: np.flatnonzero(cids == c) for c in uc}
    diffs = []
    for _ in range(OOS_BOOT_REPS):
        ix = np.concatenate([grp[c] for c in rng.choice(uc, len(uc))])
        diffs.append(d1[ix].mean() - d0[ix].mean())
    res = {"n_events": int(len(s)), "n_concepts": int(len(uc)),
           "mean_deviance_M0": float(d0.mean()), "mean_deviance_M1": float(d1.mean()),
           "deviance_diff_M1_minus_M0": float(d1.mean() - d0.mean()),
           "deviance_diff_ci95_concept_bootstrap": [float(np.quantile(diffs, .025)), float(np.quantile(diffs, .975))],
           "spearman_M0": float(stats.spearmanr(s.mu0, y).statistic),
           "spearman_M1": float(stats.spearmanr(s.mu1, y).statistic),
           "note": "grouped-by-concept 5-fold out-of-fold predictions; descriptive predictive check, not the "
                   "inferential test (concept x e FE cannot be used for unseen concepts)"}
    return s, res


# from assemble_out.run(): same sample as the models, indexed like the input table
t = time.time()
df = df_events
s = models.primary_sample(df)
s.index = df[(df.arm == "main") & (df.fold == "screen") & df.kw5 & df.MAIN].dropna(
    subset=["A_cont", "CT"] + models.EVENT_CONTROLS + models.SECONDARY_EXTRA).index
so, oos_res = oos(s)
(RESULTS / "oos_check.json").write_text(json.dumps(oos_res, indent=1))
timings[12] = round(time.time() - t, 1)
logger.info(f"oos: {oos_res}")

## Results
Anchoring (`A_cont`) and co-transfer (`CT`) effects as incidence-rate ratios per SD with 95% CIs for every model the
demo fitted, next to the full-run (1,740-event) values from the artifact's `results/d2_models_table.csv` and
`oos_check.json`. IRR > 1 means more newcomer uptake in the host after the entry.

In [ ]:
# full-run reference values (artifact results/d2_models_table.csv, oos_check.json; 1,740 events, 184 concepts)
FULL = {("M1", "A_cont"): (1.386, 0.968, 1.985, 452, 77), ("M1", "CT"): (1.037, 0.717, 1.500, 452, 77),
        ("M1_secondary", "A_cont"): (1.300, 1.164, 1.452, 1544, 140), ("M1_secondary", "CT"): (1.051, 0.916, 1.206, 1544, 140),
        ("M1_coarse_cx2y", "A_cont"): (1.101, 0.831, 1.460, 680, 98), ("M1_coarse_cx2y", "CT"): (0.825, 0.652, 1.042, 680, 98),
        ("M1_c_plus_dxe", "A_cont"): (1.215, 1.054, 1.401, 1016, 129), ("M1_c_plus_dxe", "CT"): (0.905, 0.771, 1.062, 1016, 129)}
FULL_DECISION = {"M1": "NEITHER", "M1_secondary": "GRAFTING", "M1_coarse_cx2y": "NEITHER", "M1_c_plus_dxe": "GRAFTING"}

tab = pd.read_csv(RESULTS / "d2_models_table.csv")
tab["full_irr_sd"] = [FULL.get((m, v), (np.nan,))[0] for m, v in zip(tab.model, tab["var"])]
show = tab[["model", "spec", "y", "var", "irr_sd", "irr_sd_lo", "irr_sd_hi", "p", "p_wild", "n", "G", "full_irr_sd"]]
pd.set_option("display.width", 200)
print(show.round(3).to_string(index=False))

print("\nDecisions (demo vs full run):")
dec_key = {"M1": "decision_primary", "M1_secondary": "decision_secondary",
           "M1_coarse_cx2y": "decision_M1_coarse_cx2y", "M1_c_plus_dxe": "decision_M1_c_plus_dxe"}
for m, k in dec_key.items():
    d = res[k]
    print(f"  {m:16s} demo: {d['reading'] if d else 'not estimable':9s} "
          f"(Holm p A={d['holm_p']['A_cont']:.3g}, CT={d['holm_p']['CT']:.3g})  | full run: {FULL_DECISION[m]}"
          if d else f"  {m:16s} demo: not estimable | full run: {FULL_DECISION[m]}")
print("\nEST_bin LPM (primary):", {k: res["EST_bin_LPM"][k] for k in ["n", "coef", "p"]})
print("OOS check demo :", {k: oos_res[k] for k in ["n_events", "deviance_diff_M1_minus_M0",
                                                    "deviance_diff_ci95_concept_bootstrap", "spearman_M0", "spearman_M1"]})
full_oos = data["metadata"]["oos"]
print("OOS check full :", {k: full_oos[k] for k in ["n_events", "deviance_diff_M1_minus_M0",
                                                    "deviance_diff_ci95_concept_bootstrap", "spearman_M0", "spearman_M1"]})
print("\nstage timings (s):", timings, "| total", round(time.time() - t0, 1))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# (a) forest plot: IRR per SD of A_cont and CT, demo vs full run, for the four Y_strict M1 specifications
ax = axes[0]
mods = ["M1", "M1_secondary", "M1_coarse_cx2y", "M1_c_plus_dxe"]
labels = {"M1": "primary\nconcept×e + d×e", "M1_secondary": "secondary (co-primary)\nconcept + e + d",
          "M1_coarse_cx2y": "concept×2y + d×e", "M1_c_plus_dxe": "concept + d×e"}
for j, var in enumerate(["A_cont", "CT"]):
    col = ["#1f77b4", "#d62728"][j]
    for i, m in enumerate(mods):
        yy = i * 3 + j
        r = tab[(tab.model == m) & (tab["var"] == var)]
        if len(r):
            r = r.iloc[0]
            ax.errorbar(r.irr_sd, yy + 0.2, xerr=[[r.irr_sd - r.irr_sd_lo], [r.irr_sd_hi - r.irr_sd]], fmt="o",
                        color=col, capsize=3, label=f"{var} demo" if i == 0 else None)
        f = FULL[(m, var)]
        ax.errorbar(f[0], yy - 0.2, xerr=[[f[0] - f[1]], [f[2] - f[0]]], fmt="s", color=col, alpha=0.4, capsize=3,
                    label=f"{var} full run" if i == 0 else None)
ax.axvline(1, color="k", lw=0.8, ls="--")
ax.set_yticks([i * 3 + 0.5 for i in range(len(mods))])
ax.set_yticklabels([labels[m] for m in mods], fontsize=8)
ax.set_xscale("log")
ax.set_xlabel("IRR per SD (95% CI, log scale)")
ax.set_title("(a) Anchoring vs co-transfer: Y_strict PPML")
ax.legend(fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=2)
ax.invert_yaxis()

# (b) raw relation: newcomer uptake rate vs anchoring, coloured by co-transfer
ax = axes[1]
rate = df_events.Y_strict / df_events.n_entry_papers
sc = ax.scatter(df_events.A_cont, rate + 0.01, c=df_events.CT, cmap="viridis", s=25, edgecolor="k", lw=0.3)
ax.set_yscale("log")
ax.set_xlabel("A_cont (partners' pre-entry host share)")
ax.set_ylabel("Y_strict / n_entry_papers (+0.01)")
ax.set_title("(b) Newcomer uptake vs anchoring (demo events)")
plt.colorbar(sc, ax=ax, label="CT (origin co-transfer share)")

# (c) out-of-sample predictions M0 vs M1
ax = axes[2]
lim = [0.05, max(so.Y_strict.max(), so.mu1.max(), so.mu0.max()) * 1.5]
ax.scatter(so.mu0 + 0.05, so.Y_strict + 0.05, s=18, alpha=0.6, label=f"M0 controls (ρ={oos_res['spearman_M0']:.2f})")
ax.scatter(so.mu1 + 0.05, so.Y_strict + 0.05, s=18, alpha=0.6, marker="x",
           label=f"M1 + A_cont + CT (ρ={oos_res['spearman_M1']:.2f})")
ax.plot(lim, lim, "k--", lw=0.8)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("out-of-fold predicted Y_strict (+0.05)")
ax.set_ylabel("observed Y_strict (+0.05)")
ax.set_title("(c) Grouped out-of-sample check")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### Reading the demo output
* **Stage 0** reproduces the pre-registration hash of the full run bit for bit, so the analysis above is the frozen one.
* **Stage 7** runs all pre-declared PPML specifications. The in-house IRLS estimator matches `pyfixest.fepois`
  (coefficients within 1e-4, SEs within 0.1%). With only 20 concept clusters the CIs are several times wider than in
  the full run, and every specification reads **NEITHER** here. That is expected at this size: the full run's power
  simulation puts the co-primary MDE at IRR/SD 1.15 for a 93-concept / 1,100-event design, and this demo has 20 concepts.
  So the demo does not contradict the headline. The co-primary GRAFTING result (A IRR/SD 1.30 [1.16, 1.45]) rests on
  the 1,740 events / 140 clusters of the full screen fold. The thin-cells fallback triggers here too (G = 20 < 50).
* **Stage 12**: in the full run, adding `A_cont` + `CT` improves out-of-sample deviance only slightly (−0.23, CI
  crossing 0). With 100 events the same quantity is far noisier.

To reproduce the full numbers, run the original pipeline (`python method.py`) on the hydrated corpus. Raising `N_EVENTS`
cannot help because the demo file holds only 100 events.